# 01 — Exploratory data analysis

**Goal:** Understand what fraud looks like in PaySim before modeling.

**Outputs of this notebook:**
- A clear mental model of the dataset (size, schema, class balance, fraud patterns)
- A few key plots saved to `reports/figures/`
- Five sentences in the README about what fraud looks like

**Stop signal:** You can confidently answer: which transaction types contain fraud? what's the class balance? are there obvious leakage signals?

## Setup

In [ ]:
# Reload changes to src/ modules without restarting the kernel.
%load_ext autoreload
%autoreload 2

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fraud_mlops.config import FIGURES_DIR, RANDOM_SEED, ensure_dirs
from fraud_mlops.data import load_paysim

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
ensure_dirs()

np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid")

print(f"Figures will be saved to: {FIGURES_DIR}")

## Load the data

If this fails with `FileNotFoundError`, run `bash scripts/download_data.sh` first.

In [ ]:
# drop_leaky=False here so we can SEE the leaky columns first.
# We'll drop them in the modeling notebook.
df = load_paysim(drop_leaky=False)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Schema and dtypes — first thing to check on any new dataset.
df.info()

In [ ]:
# Numeric summary. Look for: zero-inflation, extreme max values, suspicious zeros in balances.
df.describe()

## Class balance — the central fact about this dataset

In [ ]:
fraud_rate = df["isFraud"].mean()
n_fraud = df["isFraud"].sum()
n_legit = (~df["isFraud"].astype(bool)).sum()

print(f"Total transactions: {len(df):,}")
print(f"Fraudulent:         {n_fraud:,} ({fraud_rate:.4%})")
print(f"Legitimate:         {n_legit:,} ({1 - fraud_rate:.4%})")
print()
print(f"Trivial baseline (always predict 'legit'): {1 - fraud_rate:.4%} accuracy")
print("This is why we don't use accuracy as a metric for this problem.")

## Where does fraud actually live? Breakdown by transaction type

PaySim has 5 transaction types. Fraud is not uniformly distributed across them — this is one of the most important observations in the entire EDA.

In [ ]:
type_breakdown = (
    df.groupby("type")
    .agg(
        total=("isFraud", "size"),
        frauds=("isFraud", "sum"),
    )
    .assign(fraud_rate=lambda x: x["frauds"] / x["total"])
    .sort_values("frauds", ascending=False)
)
type_breakdown

**Key observation:** Fraud only occurs in `TRANSFER` and `CASH_OUT` transactions. CASH_IN, DEBIT, and PAYMENT have zero fraud. This means a useful first filter is: only score `TRANSFER` and `CASH_OUT` events. Mention this in your design discussion — it's a real-world cost saver.

## The leakage check — `isFlaggedFraud`

PaySim's documentation says: *the simulator flags a transfer as `isFlaggedFraud=1` whenever an attempt is made to transfer more than 200,000 in a single transaction.* This is a rule-based label, not a real signal. If we use it as a feature, our model trivially learns the rule.

In [ ]:
flag_breakdown = pd.crosstab(df["isFlaggedFraud"], df["isFraud"])
flag_breakdown.columns = ["actual_legit", "actual_fraud"]
flag_breakdown.index.name = "isFlaggedFraud"
print(flag_breakdown)
print()
print(f"Of the {df['isFlaggedFraud'].sum()} flagged transactions, all are actual fraud,")
print("but they're a tiny fraction of total fraud. This is a leak — drop it.")

## Amount distributions — fraud vs legit

We expect fraudulent transactions to look different in amount. Plotting on log scale because amounts span many orders of magnitude.

In [ ]:
# Restrict to TRANSFER/CASH_OUT for a fair comparison (no fraud exists elsewhere).
fraud_types = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
fraud_types["log_amount"] = np.log10(fraud_types["amount"] + 1)

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(
    data=fraud_types,
    x="log_amount",
    hue="isFraud",
    bins=60,
    stat="density",
    common_norm=False,
    alpha=0.5,
    ax=ax,
)
ax.set_title("Transaction amount distribution (TRANSFER + CASH_OUT)")
ax.set_xlabel("log10(amount + 1)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_amount_by_class.png", dpi=120)
plt.show()

## The balance signature of fraud

Hypothesis: a fraudster drains the source account. So we expect `oldbalanceOrg > 0` and `newbalanceOrig == 0` for fraud. Let's check.

In [ ]:
df["drains_origin"] = (df["oldbalanceOrg"] > 0) & (df["newbalanceOrig"] == 0)
drain_breakdown = pd.crosstab(df["drains_origin"], df["isFraud"], normalize="index")
drain_breakdown.columns = ["P(legit | drain)", "P(fraud | drain)"]
print(drain_breakdown)
print()
print(
    "If P(fraud | drain) is way higher than the base rate, this is a real signal.\n"
    f"Base fraud rate: {fraud_rate:.4%}"
)

## Time pattern

PaySim's `step` is hours since simulation start (1..744 ≈ 1 month). Does fraud cluster at certain hours?

In [ ]:
fraud_per_step = df.groupby("step")["isFraud"].agg(["sum", "size"])
fraud_per_step["rate"] = fraud_per_step["sum"] / fraud_per_step["size"]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(fraud_per_step.index, fraud_per_step["size"], lw=0.8)
axes[0].set_ylabel("transactions / hour")
axes[0].set_title("Volume and fraud rate over time")
axes[1].plot(fraud_per_step.index, fraud_per_step["rate"], lw=0.8, color="crimson")
axes[1].set_ylabel("fraud rate")
axes[1].set_xlabel("step (hours)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_time_pattern.png", dpi=120)
plt.show()

## What you should now know

Fill in the README's "What fraud looks like in this data" section with five sentences based on the above. A starter template:

1. Fraud rate is ~0.13% — extreme imbalance; standard accuracy is meaningless.
2. All fraud is in TRANSFER and CASH_OUT types; other types can be filtered out upstream.
3. `isFlaggedFraud` is a rule-based leak and must be dropped.
4. Fraud transactions tend to drain the origin account (oldbalanceOrg > 0, newbalanceOrig = 0).
5. [Your own observation about timing or amount distributions.]

**Next:** open `notebooks/02_baseline_model.ipynb` and train your first model.